# HG Pretrain Runner

This notebook trains the Heterogeneous Graph Neural Network (HeteroGNN) via contrastive learning to generate biologically informed drug embeddings.

The trained embeddings are consumed by XPert as a static drug representation (see `../xpert/run.ipynb`).

---

## Directory Structure

```
hg/
├── src/                    # Source code
│   ├── model_hg.py         #   HeteroGNN model definition
│   ├── pretrain_hg.py      #   Training entry point
│   └── utils_hg.py         #   Utility functions (contrastive loss, subgraph, etc.)
├── data/                   # Input: graph node/edge data
│   ├── all_gene_node_feat_19392.npy
│   ├── all_drug_node_feat_8981.npy
│   ├── PPI_edge_index_all_901260_pairs_weighted.npy
│   ├── DTI_edge_index_all_12890_pairs.npy
│   ├── DDS_edge_index_all_287834_pairs_weighted.npy
│   └── 978_gene_node_idx.npy
├── model/                  # Output: model checkpoints + training logs
│   └── <timestamp>/
│       ├── <timestamp>.log
│       ├── global_fold_early_stop.pth
│       ├── phase1_drug_embeddings.npy
│       └── phase1_gene_embeddings.npy
├── embeddings/             # Output: final embeddings
│   ├── HG_drug_embeddings.npy   #   (8981, 256) — used by XPert
│   └── article/                 #   Author-provided reference embeddings
│       └── HG_drug_embeddings.npy
└── run.sh                  # Shell script entry
```

In [ ]:
import os
import subprocess

HG_ROOT = os.getcwd()

def run_script(script_path, *args, cwd=None):
    """Run a shell script with optional arguments."""
    if cwd is None:
        cwd = HG_ROOT
    cmd = ['bash', script_path] + list(args)
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=cwd, text=True)
    if result.returncode != 0:
        print(f"Error: Script exited with code {result.returncode}")
    return result.returncode

---

## Step 1: Verify Input Data

Check that all required data files exist and inspect their shapes.

### Required Data Files

| File | Path | Size | Format | Description |
|------|------|------|--------|-------------|
| Gene node features | `data/all_gene_node_feat_19392.npy` | 76 MB | `np.ndarray` shape `(19392, 512)` float32 | Feature vectors for 19,392 gene nodes |
| Drug node features | `data/all_drug_node_feat_8981.npy` | 36 MB | `np.ndarray` shape `(8981, 512)` float32 | Feature vectors for 8,981 drug nodes |
| PPI edges | `data/PPI_edge_index_all_901260_pairs_weighted.npy` | 21 MB | `dict` with `edge_index` + `edge_attr` | 901,260 weighted protein-protein interaction edges (gene-gene) |
| DTI edges | `data/DTI_edge_index_all_12890_pairs.npy` | 202 KB | `dict` with `edge_index` | 12,890 drug-target interaction edges (drug-gene) |
| DDS edges | `data/DDS_edge_index_all_287834_pairs_weighted.npy` | 6.6 MB | `dict` with `edge_index` + `edge_attr` | 287,834 weighted drug-drug similarity edges (drug-drug) |
| L1000 gene index | `data/978_gene_node_idx.npy` | 7.8 KB | `np.ndarray` shape `(978,)` int | Indices of 978 landmark genes (for relation matrix / DTI inference) |

### Data Structure Details

**Node features**: Each row is a 512-dimensional feature vector. Gene features are derived from gene expression profiles; drug features from chemical fingerprints.

**Edge files** (`.npy` dicts): Loaded via `np.load(path, allow_pickle=True).item()`. The dict contains:
- `'edge_index'`: `np.ndarray` shape `(2, num_edges)`, row 0 = source indices, row 1 = target indices
- `'edge_attr'` (weighted files only): `np.ndarray` shape `(num_edges,)`, edge weights

**HeteroData construction** (in `src/pretrain_hg.py`):
```
HeteroData:
  gene.x = all_gene_node_feat_19392          # (19392, 512)
  drug.x = all_drug_node_feat_8981           # (8981, 512)
  (gene, PPI, gene).edge_index + edge_attr   # gene<->gene
  (drug, DTI, gene).edge_index               # drug->gene (reverse also added)
  (drug, DDS, drug).edge_index + edge_attr   # drug<->drug
```

In [ ]:
import numpy as np

data_dir = os.path.join(HG_ROOT, 'data')

required_files = {
    'all_gene_node_feat_19392.npy': 'Gene node features',
    'all_drug_node_feat_8981.npy': 'Drug node features',
    'PPI_edge_index_all_901260_pairs_weighted.npy': 'PPI edges',
    'DTI_edge_index_all_12890_pairs.npy': 'DTI edges',
    'DDS_edge_index_all_287834_pairs_weighted.npy': 'DDS edges',
    '978_gene_node_idx.npy': 'L1000 gene index',
}

print('=== Data File Verification ===')
all_ok = True
for fname, desc in required_files.items():
    fpath = os.path.join(data_dir, fname)
    if os.path.exists(fpath):
        arr = np.load(fpath, allow_pickle=True)
        if hasattr(arr, 'shape'):
            print(f'  [OK] {fname}: shape={arr.shape}, dtype={arr.dtype}')
        else:
            print(f'  [OK] {fname}: type={type(arr)}')
    else:
        print(f'  [MISSING] {fname} ({desc})')
        all_ok = False

if all_ok:
    print('\nAll data files present!')
else:
    print('\nSome data files are missing!')

---

## Step 2: Train HeteroGNN

Train the HeteroGNN model using contrastive learning (InfoNCE loss).

**Script**: `run.sh`  
**Entry**: `src/pretrain_hg.py`  

### Key Hyperparameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `--include_edges` | `PPI DTI DDS` | Which edge types to include in the graph |
| `--num_layers` | 3 | Number of HeteroConv layers |
| `--hidden_dim` | 256 | Hidden dimension size |
| `--out_dim` | 256 | Output dimension size |
| `--num_neighbors` | `10 5 3` | Neighbor sampling sizes per layer |
| `--batch_size` | 1 | Number of drug seed nodes per batch |
| `--epochs` | 300 | Maximum training epochs |
| `--patience` | 30 | Early stopping patience |
| `--conseq_patience` | 3 | Consecutive loss increase patience |
| `--turn_patience` | 10 | Total loss increase patience |
| `--num_neg_samples` | 3 | Negative samples per positive edge |
| `--temperature` | 0.5 | InfoNCE temperature |
| `--device` | `cuda:0` | Training device |

In [ ]:
run_script('run.sh')

---

## Step 3: Verify Output Embeddings

After training, verify that the drug embeddings were saved correctly.

### Output Files

| File | Path | Format | Description |
|------|------|--------|-------------|
| HG drug embeddings | `embeddings/HG_drug_embeddings.npy` | `np.ndarray` shape `(8981, 256)` float32 | 256-dim embedding for each drug node, used by XPert |
| Phase 1 embeddings | `model/<timestamp>/phase1_drug_embeddings.npy` | same | Backup copy in log directory |
| Phase 1 gene embeddings | `model/<timestamp>/phase1_gene_embeddings.npy` | `np.ndarray` shape `(19392, 256)` | Gene embeddings (for relation matrix) |
| Best model checkpoint | `model/<timestamp>/global_fold_early_stop.pth` | PyTorch checkpoint | Contains `model_state_dict` + `optimizer_state_dict` + `epoch` |

In [ ]:
embed_path = os.path.join(HG_ROOT, 'embeddings', 'HG_drug_embeddings.npy')

if os.path.exists(embed_path):
    embed = np.load(embed_path, allow_pickle=True)
    print(f'HG drug embeddings: shape={embed.shape}, dtype={embed.dtype}')
    print(f'  Value range: [{embed.min():.4f}, {embed.max():.4f}]')
    print(f'  Mean: {embed.mean():.4f}, Std: {embed.std():.4f}')
else:
    print(f'Embeddings not found at {embed_path}')
    print('Please run Step 2 first.')

---

## Step 4: Compare with Article Embeddings (Optional)

Compare the trained embeddings with the author-provided reference embeddings.

In [ ]:
article_path = os.path.join(HG_ROOT, 'embeddings', 'article', 'HG_drug_embeddings.npy')

if os.path.exists(embed_path) and os.path.exists(article_path):
    article_embed = np.load(article_path, allow_pickle=True)
    print(f'Article embeddings: shape={article_embed.shape}, dtype={article_embed.dtype}')
    print(f'Trained embeddings: shape={embed.shape}, dtype={embed.dtype}')
    print()
    if embed.shape == article_embed.shape:
        from numpy.linalg import norm
        cos_sim = np.sum(embed * article_embed, axis=1) / (norm(embed, axis=1) * norm(article_embed, axis=1) + 1e-8)
        print(f'Cosine similarity (trained vs article):')
        print(f'  Mean: {cos_sim.mean():.4f}')
        print(f'  Std:  {cos_sim.std():.4f}')
        print(f'  Min:  {cos_sim.min():.4f}')
        print(f'  Max:  {cos_sim.max():.4f}')
    else:
        print('Shape mismatch — cannot compare directly.')
else:
    print('Article embeddings not found or trained embeddings not available.')

---

## Step 5: Advanced Options

### Resume from a Saved Checkpoint

Use `--global_saved_model` to load a previously trained model and skip Phase 1 training.

In [ ]:
# Resume from a checkpoint
# CHECKPOINT = 'model/<TIMESTAMP>/global_fold_early_stop.pth'
# run_script('run.sh', f'--global_saved_model={CHECKPOINT}')

### Infer DTI Scores

Use `--infer_dti_scores` to compute the full drug-target interaction score matrix after training.

In [ ]:
# run_script('run.sh', '--infer_dti_scores True')

### Compute Relation Matrix

Use `--output_relation_matrix` to compute the cosine similarity matrix between drug and gene embeddings.

In [ ]:
# run_script('run.sh', '--output_relation_matrix True')

---

## Data Flow Summary

```
Input (data/):
  all_gene_node_feat_19392.npy --|
  all_drug_node_feat_8981.npy ---|
  PPI_edge_index_*.npy ----------|-> HeteroGNN (contrastive learning)
  DTI_edge_index_*.npy ----------|         |
  DDS_edge_index_*.npy ----------|         v
                                   embeddings/HG_drug_embeddings.npy (8981, 256)
                                        |
                                        v
                                   ../xpert/ (consumed as static drug features)

Output:
  embeddings/HG_drug_embeddings.npy  — Drug embeddings (used by XPert)
  model/<timestamp>/                  — Training logs, checkpoints, phase1 embeddings
```